# ForestClustering: hyperparameter effects

Systematic study of how each `ForestClusterer` parameter affects:

| Metric | What we measure |
|--------|----------------|
| **ARI** | Adjusted Rand Index vs known labels |
| **Std(ARI)** | Variance across random_state — stability |
| **Time** | Wall time of `fit_predict`, seconds |
| **Memory** | Embedding n×L size vs matrix n×n |

Dataset: synthetic, 5 clusters, 12 mixed features (continuous, binary,
categorical, correlated duplicates, noise).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math
import multiprocessing
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
from forest_clustering import ForestClusterer

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120})

N_CLUSTERS = 5
print("OK")

## Dataset

In [ ]:
def make_dataset(n_samples, n_clusters=5, outlier_fraction=0.0, seed=42):
    """Synthetic dataset with known clusters and mixed feature types.

    Structure:
      cont_1/2/3   — informative continuous
      binary_1/2   — informative binary
      cat_1/2      — informative categorical (4 and 5 values)
      corr_1/2     — true duplicates: corr_k ≈ cont_k + ε (sigma=0.05)
      noise_1/2/3  — pure noise
    """
    rng = np.random.default_rng(seed)
    n_out   = int(n_samples * outlier_fraction)
    n_core  = n_samples - n_out

    sizes = rng.multinomial(n_core, [1 / n_clusters] * n_clusters)

    # Well-separated centres (min distance 4)
    centers = rng.uniform(-7, 7, (n_clusters, 3))
    for _ in range(300):
        ok = all(np.linalg.norm(centers[i] - centers[j]) >= 4
                 for i in range(n_clusters) for j in range(i + 1, n_clusters))
        if ok:
            break
        centers = rng.uniform(-7, 7, (n_clusters, 3))

    cont_parts, labels = [], []
    for k, n_k in enumerate(sizes):
        labels.extend([k] * n_k)
        cont_parts.append(rng.normal(centers[k], 0.9, (n_k, 3)))

    X_cont = np.vstack(cont_parts)
    y_core = np.array(labels)

    # Binary: P(1) monotone across clusters
    X_bin = np.column_stack([
        rng.binomial(1, 0.1 + 0.8 * y_core / (n_clusters - 1)),
        rng.binomial(1, 0.9 - 0.8 * y_core / (n_clusters - 1)),
    ]).astype(float)

    # Categorical: dominant class = k % n_cats
    def cat_col(n_cats, dominant_strength=0.7):
        cols = []
        for k, n_k in enumerate(sizes):
            probs = np.full(n_cats, (1 - dominant_strength) / (n_cats - 1))
            probs[k % n_cats] = dominant_strength
            cols.append(rng.choice(n_cats, n_k, p=probs))
        return np.concatenate(cols)

    X_cat = np.column_stack([cat_col(4), cat_col(5)])

    # Correlated duplicates: corr_k ≈ cont_k + ε (Spearman ≈ 0.999)
    X_corr = X_cont[:, :2] + rng.normal(0, 0.05, (n_core, 2))

    # Pure noise
    X_noise = rng.normal(0, 4, (n_core, 3))

    if n_out > 0:
        X_cont  = np.vstack([X_cont,  rng.uniform(-22, 22, (n_out, 3))])
        X_bin   = np.vstack([X_bin,   rng.binomial(1, 0.5, (n_out, 2)).astype(float)])
        X_cat   = np.vstack([X_cat,   np.column_stack([rng.integers(0, 4, n_out),
                                                        rng.integers(0, 5, n_out)])])
        X_corr  = np.vstack([X_corr,  rng.uniform(-22, 22, (n_out, 2))])
        X_noise = np.vstack([X_noise, rng.normal(0, 4, (n_out, 3))])
        y_core  = np.concatenate([y_core, np.full(n_out, -1)])

    X = np.hstack([X_cont, X_bin, X_cat.astype(float), X_corr, X_noise])
    perm = rng.permutation(n_samples)
    X, y = X[perm], y_core[perm]

    COLS = ['cont_1','cont_2','cont_3',
            'binary_1','binary_2',
            'cat_1','cat_2',
            'corr_1','corr_2',
            'noise_1','noise_2','noise_3']

    df = pd.DataFrame(X, columns=COLS)
    df['cat_1'] = df['cat_1'].astype(int).map({0:'A',1:'B',2:'C',3:'D'})
    df['cat_2'] = df['cat_2'].astype(int).map({0:'v',1:'w',2:'x',3:'y',4:'z'})
    df['binary_1'] = df['binary_1'].astype(int)
    df['binary_2'] = df['binary_2'].astype(int)
    return df, y

N = 2_500                                    # base size for sweeps
df0, y0 = make_dataset(N, N_CLUSTERS, seed=42)
COLS = df0.columns.tolist()
D = len(COLS)
print(f"n={N}, d={D}, clusters: {np.unique(y0, return_counts=True)}")

# Fixed downstream clusterer — we measure only FC parameter effects
INNER = lambda: KMeans(n_clusters=N_CLUSTERS, n_init='auto', random_state=0)

In [ ]:
def run_sweep(param, values, df, y_true, base, n_seeds=3):
    """Sweep one parameter, others = base. Returns a DataFrame of results."""
    records = []
    for v in values:
        aris, times = [], []
        for s in range(n_seeds):
            clf = ForestClusterer(**{**base, param: v},
                                  clusterer=INNER(), random_state=s)
            t0 = time.perf_counter()
            lbl = clf.fit_predict(df)
            elapsed = time.perf_counter() - t0
            mask = lbl >= 0
            ari = adjusted_rand_score(y_true[mask], lbl[mask]) if mask.sum() > 10 else 0.0
            aris.append(ari)
            times.append(elapsed)
        records.append({
            'value': v,
            'ari_mean': np.mean(aris), 'ari_std': np.std(aris),
            'time_mean': np.mean(times), 'time_std': np.std(times),
        })
    return pd.DataFrame(records)

# Recommended base parameters
BASE = dict(
    n_iterations=200,
    n_bins=3,
    n_features='sqrt',
    quantile_cuts=True,
    corr_threshold=0.9,
    corr_sample_size=N,
    n_jobs=-1,
)
print("Helper ready. BASE =", BASE)

## 1. `n_iterations` — main lever for quality and speed

Each iteration adds one column to the n×L embedding and refines the distance estimate.

**Expected behaviour:**
- ARI grows and plateaus — "law of large numbers" for random partitions
- Std(ARI) drops — more iterations → less influence of random_state
- Time grows **strictly linearly** (iterations are independent, parallelised uniformly)

In [ ]:
L_vals = [10, 20, 50, 100, 150, 200, 300, 400, 500]
df_L = run_sweep('n_iterations', L_vals, df0, y0, BASE, n_seeds=5)
df_L['emb_mb'] = [N * l * 8 / 1e6 for l in L_vals]
df_L.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ARI with ±std band
ax = axes[0]
ax.plot(df_L.value, df_L.ari_mean, 'o-', color='#2196F3', lw=2.5, ms=7, zorder=3)
ax.fill_between(df_L.value,
                df_L.ari_mean - df_L.ari_std,
                df_L.ari_mean + df_L.ari_std,
                alpha=0.2, color='#2196F3', label='±1 std (5 seeds)')
ax.axvline(200, color='#888', ls='--', lw=1.2, label='default=200')
ax.set_xlabel('n_iterations (L)')
ax.set_ylabel('ARI (mean ± std)')
ax.set_title('Quality', fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)

# Time with linear fit
ax = axes[1]
ax.plot(df_L.value, df_L.time_mean, 's-', color='#FF9800', lw=2.5, ms=7, zorder=3)
ax.fill_between(df_L.value,
                df_L.time_mean - df_L.time_std,
                df_L.time_mean + df_L.time_std,
                alpha=0.2, color='#FF9800')
coeffs = np.polyfit(df_L.value, df_L.time_mean, 1)
x_fit = np.linspace(df_L.value.min(), df_L.value.max(), 200)
ax.plot(x_fit, np.polyval(coeffs, x_fit), '--', color='gray', lw=1.5,
        label=f'linear fit\n{coeffs[0]*1000:.1f} ms/iter')
ax.axvline(200, color='#888', ls='--', lw=1.2)
ax.set_xlabel('n_iterations (L)')
ax.set_ylabel('fit_predict time, s')
ax.set_title('Time (linear growth)', fontweight='bold')
ax.legend(fontsize=9)

# Stability (std ARI)
ax = axes[2]
ax.plot(df_L.value, df_L.ari_std, '^-', color='#9C27B0', lw=2.5, ms=7)
ax.axvline(200, color='#888', ls='--', lw=1.2, label='default=200')
ax.set_xlabel('n_iterations (L)')
ax.set_ylabel('Std(ARI) over 5 seeds')
ax.set_title('Stability', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('n_iterations: quality, speed, stability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

for l in [50, 100, 200, 500]:
    row = df_L[df_L.value == l].iloc[0]
    print(f"L={l:3d}: ARI={row.ari_mean:.3f} ±{row.ari_std:.3f}  "
          f"t={row.time_mean:.2f}s  emb={row.emb_mb:.1f}MB")

## 2. `n_bins` — partition granularity

Number of bins K per feature per iteration. Unique cells per iteration = K^M.

| K | K^M with M=⌈√d⌉=4 | Avg points/cell at n=2500 |
|---|---|---|
| 2 | 16 | 156 — too coarse |
| 3 | 81 | 31 — optimal |
| 4 | 256 | 10 — acceptable |
| 5 | 625 | 4 — sparse cells |
| 7 | 2401 | 1 — most cells have 1 point |

With too small K, iterations carry little information about structure. With too large K,
most cells are empty or singletons — d(i,j) ≈ 1 for all pairs.

In [ ]:
K_vals = [2, 3, 4, 5, 6, 7, 8]
df_K = run_sweep('n_bins', K_vals, df0, y0, BASE, n_seeds=3)

M_default = math.ceil(math.sqrt(D))
df_K['cells_per_iter'] = [k ** M_default for k in K_vals]
df_K['avg_pts_cell']   = N / df_K['cells_per_iter']
df_K[['value','ari_mean','ari_std','cells_per_iter','avg_pts_cell','time_mean']].round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ARI
ax = axes[0]
ax.plot(df_K.value, df_K.ari_mean, 'o-', color='#2196F3', lw=2.5, ms=8)
ax.fill_between(df_K.value, df_K.ari_mean - df_K.ari_std,
                df_K.ari_mean + df_K.ari_std, alpha=0.2, color='#2196F3')
ax.axvline(3, color='#888', ls='--', lw=1.2, label='default=3')
ax.set_xlabel('n_bins (K)')
ax.set_ylabel('ARI')
ax.set_title('Quality', fontweight='bold')
ax.set_xticks(K_vals)
ax.legend(fontsize=9)

# Cell space size
ax = axes[1]
ax_r = ax.twinx()
ax.semilogy(df_K.value, df_K.cells_per_iter, 's-', color='#FF9800', lw=2, ms=8,
             label='K^M (cells/iter)')
ax_r.plot(df_K.value, df_K.avg_pts_cell, '^--', color='#9C27B0', lw=2, ms=8,
          label='avg n/cell')
ax.set_xlabel('n_bins (K)')
ax.set_ylabel('K^M, log-scale', color='#FF9800')
ax_r.set_ylabel('Avg points per cell', color='#9C27B0')
ax.set_title(f'Cell space size (M=ceil(sqrt(d))={M_default})', fontweight='bold')
ax.axvline(3, color='#888', ls='--', lw=1.2)
ax.set_xticks(K_vals)
lines  = ax.get_lines() + ax_r.get_lines()
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, fontsize=9, loc='upper left')
ax.axhline(1, color='red', ls=':', lw=1, alpha=0.5)  # avg=1 = empty cells

# Time
ax = axes[2]
ax.plot(df_K.value, df_K.time_mean, 'o-', color='#4CAF50', lw=2.5, ms=8)
ax.fill_between(df_K.value, df_K.time_mean - df_K.time_std,
                df_K.time_mean + df_K.time_std, alpha=0.2, color='#4CAF50')
ax.axvline(3, color='#888', ls='--', lw=1.2, label='default=3')
ax.set_xlabel('n_bins (K)')
ax.set_ylabel('Time, s')
ax.set_title('Compute time', fontweight='bold')
ax.set_xticks(K_vals)
ax.legend(fontsize=9)

plt.suptitle(f'n_bins: K=3 is optimal at n={N}, d={D}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. `n_features` — feature subspace size

How many features are selected per iteration.

- **Too few** (M=1, 2) → each iteration considers too few features,
  misses joint effects
- **Too many** (M=d) → iterations are nearly identical, partition diversity is lost
- **`'sqrt'`** — analogous to RandomForest: balance of informativeness and randomness

In [ ]:
nf_options = [1, 2, 'log2', 'sqrt', 6, 0.5, 1.0]

def resolve_M(val, d):
    if val == 'sqrt':  return math.ceil(math.sqrt(d))
    if val == 'log2':  return max(1, math.ceil(math.log2(d)))
    if isinstance(val, float): return max(1, int(val * d))
    return int(val)

nf_rows = []
for val in nf_options:
    aris, times = [], []
    for s in range(3):
        clf = ForestClusterer(**{**BASE, 'n_features': val},
                              clusterer=INNER(), random_state=s)
        t0 = time.perf_counter()
        lbl = clf.fit_predict(df0)
        elapsed = time.perf_counter() - t0
        aris.append(adjusted_rand_score(y0, lbl))
        times.append(elapsed)
    m_val = resolve_M(val, D)
    label = str(val) if not isinstance(val, float) else f'{val}*d={m_val}'
    if val == 'sqrt': label = f"sqrt ≈ {m_val}"
    if val == 'log2': label = f"log2 ≈ {m_val}"
    nf_rows.append({
        'option': label, 'M': m_val,
        'ari_mean': np.mean(aris), 'ari_std': np.std(aris),
        'time_mean': np.mean(times), 'time_std': np.std(times),
    })

df_M = pd.DataFrame(nf_rows)
df_M[['option','M','ari_mean','ari_std','time_mean']].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(df_M))
def_idx = [i for i, r in df_M.iterrows() if 'sqrt' in str(r.option)][0]
colors_nf = ['#C44E52' if i == def_idx else '#2196F3' for i in x]

for ax_i, (col, label) in enumerate([('ari_mean', 'ARI'), ('time_mean', 'Time, s')]):
    ax = axes[ax_i]
    bars = ax.bar(x, df_M[col], color=colors_nf, edgecolor='white', linewidth=1.5)
    err = 'ari_std' if ax_i == 0 else 'time_std'
    ax.errorbar(x, df_M[col], yerr=df_M[err], fmt='none',
                color='black', capsize=5, lw=1.5)
    for bar, v in zip(bars, df_M[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(df_M.option, fontsize=10)
    ax.set_ylabel(label)
    ax.set_title(f'{label} vs n_features', fontweight='bold')
    if ax_i == 0:
        ax.set_ylim(0, 1.15)
    ax.legend(handles=[Patch(color='#C44E52', label='sqrt (default)'),
                        Patch(color='#2196F3', label='other')], fontsize=9)

plt.suptitle(f'd={D} features. Red = recommended n_features="sqrt"',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. `quantile_cuts` — outlier robustness

`quantile_cuts=True`: cut-points are quantiles of the data distribution.

**Mechanism**: an outlier with cont_1=100 when the normal range is [-10, 10]:
- `quantile_cuts=False`: edges = linspace(-10, 100, K+1) → all normal points
  fall into the first bin → d(i,j) ≈ 1 for all → clusters indistinguishable
- `quantile_cuts=True`: the outlier gets the extreme bin, normal points
  spread evenly across K bins — their mutual distances are unaffected

**Nuance**: on clean data `quantile_cuts=False` can give slightly higher ARI —
uniform cut-points do not lose information in the tails. At ≥3% outliers `True` wins clearly.

In [ ]:
out_fracs = [0.0, 0.03, 0.05, 0.10, 0.15, 0.20]

qc_rows = []
for frac in out_fracs:
    df_o, y_o = make_dataset(N, N_CLUSTERS, outlier_fraction=frac, seed=42)
    clean_mask = y_o >= 0
    for qc in [False, True]:
        aris = []
        for s in range(3):
            clf = ForestClusterer(**{**BASE, 'quantile_cuts': qc},
                                  clusterer=INNER(), random_state=s)
            lbl = clf.fit_predict(df_o)
            aris.append(adjusted_rand_score(y_o[clean_mask], lbl[clean_mask]))
        qc_rows.append({'frac': frac, 'qc': qc,
                         'ari_mean': np.mean(aris), 'ari_std': np.std(aris)})

df_qc = pd.DataFrame(qc_rows)
print(df_qc.pivot(index='frac', columns='qc', values='ari_mean').round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors_qc = {False: '#FF9800', True: '#2196F3'}
labels_qc  = {False: 'quantile_cuts=False', True: 'quantile_cuts=True (recommended)'}
for qc in [False, True]:
    sub = df_qc[df_qc.qc == qc]
    ax.plot(sub.frac * 100, sub.ari_mean, 'o-', color=colors_qc[qc],
            lw=2.5, ms=8, label=labels_qc[qc])
    ax.fill_between(sub.frac * 100,
                    sub.ari_mean - sub.ari_std,
                    sub.ari_mean + sub.ari_std,
                    alpha=0.15, color=colors_qc[qc])
ax.set_xlabel('Outlier fraction, %')
ax.set_ylabel('ARI on clean observations')
ax.set_title('ARI vs outlier fraction', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)

# Delta ARI (True - False)
ax = axes[1]
deltas = (df_qc[df_qc.qc == True].ari_mean.values -
          df_qc[df_qc.qc == False].ari_mean.values)
ax.bar([f'{f*100:.0f}%' for f in out_fracs], deltas,
       color=['#4CAF50' if d >= 0 else '#C44E52' for d in deltas],
       edgecolor='white', linewidth=1.5)
for i, (d, frac) in enumerate(zip(deltas, out_fracs)):
    ax.text(i, d + (0.002 if d >= 0 else -0.01),
            f'{d:+.3f}', ha='center',
            va='bottom' if d >= 0 else 'top',
            fontsize=9, fontweight='bold')
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('Outlier fraction')
ax.set_ylabel('Delta ARI (True - False)')
ax.set_title('Gain from quantile_cuts=True', fontweight='bold')

plt.suptitle('quantile_cuts: difference is small on clean data, critical with outliers',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. `corr_threshold` — correlated feature detection

Features within a correlated group are assigned weight 1/G.

**Dataset contains:**
- `corr_1 ≈ cont_1 + ε` and `corr_2 ≈ cont_2 + ε` — **true duplicates** (Spearman ≈ 0.999)
- `cont_1..3` — informative, but **not duplicates** (Spearman between clusters ≈ 0.7 — ecological correlation)

**Threshold task**: catch only true duplicates (corr_1↔cont_1),
without grouping informative features with each other.

| Threshold | What happens |
|-----------|-------------|
| 0.5–0.7 | Ecological correlation: `cont_1..3` get grouped → information loss |
| 0.9 | Only true duplicates `corr_k ↔ cont_k` → optimal |
| 1.0 (None) | No detection → duplicates waste weight |

In [ ]:
thresholds = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]

wt_records = []
ari_thr    = []

for thr in thresholds:
    # feature weights
    clf_w = ForestClusterer(**{**BASE, 'corr_threshold': thr},
                            clusterer=INNER(), random_state=42)
    clf_w.fit(df0)
    wt_records.append(dict(zip(COLS, clf_w.feature_weights_.round(3))))

    # ARI over 3 seeds
    aris = []
    for s in range(3):
        clf_a = ForestClusterer(**{**BASE, 'corr_threshold': thr},
                                clusterer=INNER(), random_state=s)
        aris.append(adjusted_rand_score(y0, clf_a.fit_predict(df0)))
    ari_thr.append({'thr': thr, 'ari_mean': np.mean(aris), 'ari_std': np.std(aris)})

# Add None (disabled)
clf_none = ForestClusterer(**{**BASE, 'corr_threshold': None},
                           clusterer=INNER(), random_state=42)
clf_none.fit(df0)
wt_records.append(dict(zip(COLS, clf_none.feature_weights_.round(3))))
aris_none = [adjusted_rand_score(y0,
    ForestClusterer(**{**BASE, 'corr_threshold': None}, clusterer=INNER(),
                    random_state=s).fit_predict(df0)) for s in range(3)]
ari_thr.append({'thr': 'None', 'ari_mean': np.mean(aris_none), 'ari_std': np.std(aris_none)})

thresholds_all = thresholds + ['None']
df_wt  = pd.DataFrame(wt_records, index=[str(t) for t in thresholds_all])
df_atr = pd.DataFrame(ari_thr)

# Spearman correlations in the data
from scipy.stats import spearmanr
sp_c1_cr1 = spearmanr(df0['cont_1'], df0['corr_1']).statistic
sp_c1_c2  = spearmanr(df0['cont_1'], df0['cont_2']).statistic
print(f"Spearman(cont_1, corr_1) = {sp_c1_cr1:.3f}  <- true duplicate")
print(f"Spearman(cont_1, cont_2) = {sp_c1_c2:.3f}  <- ecological correlation")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

# Feature weight heatmap
ax = axes[0]
# Group features by meaning
feat_order = ['cont_1','corr_1','cont_2','corr_2','cont_3',
              'binary_1','binary_2','cat_1','cat_2',
              'noise_1','noise_2','noise_3']
sns.heatmap(df_wt[feat_order].T, annot=True, fmt='.2f',
            cmap='RdYlGn', vmin=0.3, vmax=1.0, ax=ax,
            linewidths=0.4, cbar_kws={'label': 'Weight (1/G)'})
ax.set_title('Feature weights per threshold', fontweight='bold')
ax.set_xlabel('corr_threshold')
ax.set_ylabel('')

# Mark correlated pairs
for feat in ['cont_1','corr_1','cont_2','corr_2']:
    idx = feat_order.index(feat)
    ax.get_yticklabels()[idx].set_color('#C44E52')
    ax.get_yticklabels()[idx].set_fontweight('bold')
ax.set_title('Feature weights (red = correlated pairs)',
             fontweight='bold')

# ARI vs threshold
ax = axes[1]
x_thr = np.arange(len(df_atr))
bar_colors = ['#C44E52' if t == 0.9 else '#2196F3' for t in thresholds_all]
bars = ax.bar(x_thr, df_atr.ari_mean, color=bar_colors,
              edgecolor='white', linewidth=1.5)
ax.errorbar(x_thr, df_atr.ari_mean, yerr=df_atr.ari_std,
            fmt='none', color='black', capsize=5, lw=1.5)
for bar, v in zip(bars, df_atr.ari_mean):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(x_thr)
ax.set_xticklabels([str(t) for t in thresholds_all], fontsize=10)
ax.set_ylabel('ARI (mean +/- std, 3 seeds)')
ax.set_title('ARI vs corr_threshold', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.legend(handles=[Patch(color='#C44E52', label='0.9 (recommended)'),
                   Patch(color='#2196F3', label='other')], fontsize=9)

# Vertical annotation: where duplicate is caught
thr_caught = next(t for t in thresholds if t <= abs(sp_c1_cr1))
ax.annotate(f'cont\u2194corr\ndetected', color='#C44E52',
            xy=(thresholds.index(thr_caught), df_atr.iloc[thresholds.index(thr_caught)].ari_mean),
            xytext=(thresholds.index(thr_caught) - 1.3, 0.5),
            arrowprops=dict(arrowstyle='->', color='#C44E52'),
            fontsize=8)

plt.suptitle('corr_threshold: too low -> ecological correlation, too high -> duplicates missed',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nNote: corr_threshold=None (no weighting) can give high ARI")
print("when duplicates are informative features (they get double weight).")
print("Weighting matters most at high d where duplicates crowd out other informative features.")

## 6. `n_jobs` — iteration parallelism

Iterations are independent → `joblib.Parallel(backend='threading')`.
Threading is efficient: NumPy/SciPy release the GIL when calling BLAS/LAPACK.

**Expectations**: near-linear speedup up to the number of physical cores,
then plateau (Amdahl's law — Python-level overhead).

In [ ]:
N_LARGE = 25_000
df_lg, _ = make_dataset(N_LARGE, N_CLUSTERS, seed=7)
n_cpu = multiprocessing.cpu_count()
print(f"Cores: {n_cpu},  n={N_LARGE},  L=200")

jobs_grid = sorted(set([1, 2, min(4, n_cpu), min(8, n_cpu), n_cpu]))

job_rows = []
for n_jobs in jobs_grid:
    times_j = []
    for _ in range(3):    # 3 runs
        clf = ForestClusterer(**{**BASE, 'n_jobs': n_jobs, 'n_iterations': 200},
                              clusterer=INNER(), random_state=42)
        t0 = time.perf_counter()
        clf.fit(df_lg)
        times_j.append(time.perf_counter() - t0)
    label = f'{n_jobs}\n(all)' if n_jobs == n_cpu else str(n_jobs)
    job_rows.append({'n_jobs': n_jobs, 'label': label,
                     'time': np.mean(times_j), 'std': np.std(times_j)})

df_jobs = pd.DataFrame(job_rows)
t1 = df_jobs[df_jobs.n_jobs == 1]['time'].values[0]
df_jobs['speedup']    = t1 / df_jobs['time']
df_jobs['efficiency'] = df_jobs.speedup / df_jobs.n_jobs
df_jobs.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Time
ax = axes[0]
ax.bar(range(len(df_jobs)), df_jobs['time'], color='#FF9800', edgecolor='white', linewidth=1.5)
ax.errorbar(range(len(df_jobs)), df_jobs['time'], yerr=df_jobs['std'],
            fmt='none', color='black', capsize=5, lw=1.5)
for i, (_, row) in enumerate(df_jobs.iterrows()):
    ax.text(i, row['time'] + 0.01, f'{row["time"]:.2f}s',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(df_jobs)))
ax.set_xticklabels([f'n_jobs={l}' for l in df_jobs.label], fontsize=10)
ax.set_ylabel('Fit time, s')
ax.set_title(f'Time (n={N_LARGE:,}, L=200)', fontweight='bold')

# Speedup
ax = axes[1]
ax.bar(range(len(df_jobs)), df_jobs.speedup, color='#2196F3',
       edgecolor='white', linewidth=1.5, label='Actual')
ax.plot(range(len(df_jobs)), df_jobs.n_jobs, 'r--o', ms=6, lw=1.5,
        label='Ideal (linear)')
for i, (_, row) in enumerate(df_jobs.iterrows()):
    ax.text(i, row.speedup + 0.05, f'{row.speedup:.2f}x',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(range(len(df_jobs)))
ax.set_xticklabels([f'n_jobs={l}' for l in df_jobs.label], fontsize=10)
ax.set_ylabel('Speedup vs n_jobs=1')
ax.set_title('Speedup', fontweight='bold')
ax.legend(fontsize=9)

# Efficiency
ax = axes[2]
ax.bar(range(len(df_jobs)), df_jobs.efficiency * 100, color='#4CAF50',
       edgecolor='white', linewidth=1.5)
for i, (_, row) in enumerate(df_jobs.iterrows()):
    ax.text(i, row.efficiency * 100 + 0.5, f'{row.efficiency*100:.0f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.axhline(100, color='red', ls='--', lw=1, label='Ideal 100%')
ax.set_xticks(range(len(df_jobs)))
ax.set_xticklabels([f'n_jobs={l}' for l in df_jobs.label], fontsize=10)
ax.set_ylabel('Efficiency (speedup / n_jobs)')
ax.set_title(f'Parallel efficiency ({n_cpu} cores)', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('n_jobs: parallelism via joblib.threading',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Memory: embedding n×L vs distance matrix n×n

Key architectural advantage of ForestClusterer: **the n×n matrix is not stored in memory**.
It is computed on demand only when the downstream algorithm requires precomputed distances.

When using KMeans/DBSCAN directly on the embedding — only n×L is needed.

In [ ]:
n_vals = [1_000, 5_000, 10_000, 50_000, 100_000, 500_000]
L_fixed = 200

mem_rows = []
for n in n_vals:
    emb_mb  = n * L_fixed * 8 / 1e6       # int64
    dist_gb = n * n * 4 / 1e9             # float32
    mem_rows.append({
        'n': n,
        'Embedding n×L, MB': emb_mb,
        'Matrix n×n, GB':    dist_gb,
    })

df_mem = pd.DataFrame(mem_rows).set_index('n')
print(df_mem.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Linear scale (embedding)
ax = axes[0]
ax.plot(n_vals, df_mem['Embedding n×L, MB'], 'o-',
        color='#2196F3', lw=2.5, ms=8, label=f'Embedding n×{L_fixed}, MB')
for x_pt, y_pt in zip(n_vals, df_mem['Embedding n×L, MB']):
    ax.annotate(f'{y_pt:.0f}MB', (x_pt, y_pt), xytext=(5, 5),
                textcoords='offset points', fontsize=8, color='#2196F3')
ax.set_xlabel('n')
ax.set_ylabel('MB')
ax.set_title(f'Embedding n×{L_fixed} (int64) — linear growth', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# Log scale (both)
ax = axes[1]
ax.loglog(n_vals, df_mem['Embedding n×L, MB'], 'o-',
           color='#2196F3', lw=2.5, ms=8, label=f'Embedding n×{L_fixed}, MB (O(n))')
ax2 = ax.twinx()
ax2.loglog(n_vals, df_mem['Matrix n×n, GB'], 's--',
            color='#FF5722', lw=2.5, ms=8, label='Matrix n×n, GB (O(n^2))')
ax.set_xlabel('n')
ax.set_ylabel('MB (embedding)', color='#2196F3')
ax2.set_ylabel('GB (matrix)', color='#FF5722')
ax.set_title('Log scale: O(n) vs O(n^2)', fontweight='bold')
lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [l.get_label() for l in lines], fontsize=9, loc='upper left')

# Mark "unavailable"
for n, gbs in zip(n_vals, df_mem['Matrix n×n, GB']):
    if gbs > 32:
        ax2.annotate('> 32 GB warning', (n, gbs), xytext=(5, -15),
                     textcoords='offset points', fontsize=7, color='#FF5722')

plt.suptitle('Memory: embedding n×L grows linearly, matrix n×n quadratically',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nDownstream recommendations by n:")
print("  n <= 5K:  any algorithm (AgglClust, HDBSCAN precomputed — OK)")
print("  n <= 50K: KMeans on embedding (or DBSCAN metric='hamming')")
print("  n > 50K:  MiniBatchKMeans on embedding, no distance matrix")

## 8. Sensitivity summary

How much does Δ ARI drop with a "bad" value of each parameter?
Calculated relative to baseline with recommended parameters.

In [ ]:
# Baseline ARI (recommended parameters, 5 seeds)
ari_base_list = [adjusted_rand_score(y0,
    ForestClusterer(**BASE, clusterer=INNER(), random_state=s).fit_predict(df0))
    for s in range(5)]
ari_base = np.mean(ari_base_list)
print(f"Baseline ARI: {ari_base:.3f} +/- {np.std(ari_base_list):.3f}")

# Collect sensitivity
sens = []
# n_iterations: minimum value
row_L_min = df_L[df_L.value == 10].iloc[0]
sens.append({'Parameter': 'n_iterations=10', 'Recommended': '200',
             'ΔARI': row_L_min.ari_mean - ari_base,
             'Time impact': 'proportional to L'})

# n_iterations: maximum from sweep
row_L_max = df_L[df_L.value == 500].iloc[0]
sens.append({'Parameter': 'n_iterations=500', 'Recommended': '200',
             'ΔARI': row_L_max.ari_mean - ari_base,
             'Time impact': f'+{500/200*100-100:.0f}%'})

# n_bins=2
row_K2 = df_K[df_K.value == 2].iloc[0]
sens.append({'Parameter': 'n_bins=2', 'Recommended': '3',
             'ΔARI': row_K2.ari_mean - ari_base,
             'Time impact': 'negligible'})

# n_bins=8
row_K8 = df_K[df_K.value == 8].iloc[0]
sens.append({'Parameter': 'n_bins=8', 'Recommended': '3',
             'ΔARI': row_K8.ari_mean - ari_base,
             'Time impact': 'negligible'})

# n_features=1
row_M1 = df_M[df_M.M == 1].iloc[0]
sens.append({'Parameter': 'n_features=1', 'Recommended': 'sqrt',
             'ΔARI': row_M1.ari_mean - ari_base,
             'Time impact': 'faster (fewer features)'})

# quantile_cuts=False at 10% outliers
r_qf = df_qc[(df_qc.frac == 0.10) & (df_qc.qc == False)].ari_mean.values[0]
r_qt = df_qc[(df_qc.frac == 0.10) & (df_qc.qc == True)].ari_mean.values[0]
sens.append({'Parameter': 'quantile_cuts=False (+10% outliers)', 'Recommended': 'True',
             'ΔARI': r_qf - r_qt,
             'Time impact': 'negligible'})

# corr_threshold=0.5
row_thr05 = df_atr[df_atr.thr == 0.5].iloc[0]
row_thr09 = df_atr[df_atr.thr == 0.9].iloc[0]
sens.append({'Parameter': 'corr_threshold=0.5', 'Recommended': '0.9',
             'ΔARI': row_thr05.ari_mean - row_thr09.ari_mean,
             'Time impact': 'negligible'})

df_sens = pd.DataFrame(sens).set_index('Parameter')
df_sens['ΔARI'] = df_sens['ΔARI'].round(3)
df_sens

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

params_s = df_sens.index.tolist()
deltas_s = df_sens['ΔARI'].values.astype(float)
colors_s  = ['#C44E52' if d < -0.05 else
              '#FF9800' if d < 0 else
              '#4CAF50' for d in deltas_s]

bars = ax.barh(params_s[::-1], deltas_s[::-1], color=colors_s[::-1],
               edgecolor='white', linewidth=1.5, height=0.6)
ax.axvline(0, color='black', lw=1.5)

for bar, val in zip(bars, deltas_s[::-1]):
    offset = -0.005 if val < 0 else 0.003
    ha = 'right' if val < 0 else 'left'
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f'{val:+.3f}', ha=ha, va='center',
            fontsize=10, fontweight='bold')

ax.set_xlabel('ΔARI (bad value − recommended)', fontsize=11)
ax.set_title('Sensitivity: how critical is a hyperparameter mistake?',
             fontsize=12, fontweight='bold')
ax.set_xlim(min(deltas_s) * 1.4, max(0.1, max(deltas_s) * 2))

legend_elems = [
    Patch(color='#C44E52', label='Critical (ΔARI < -0.05)'),
    Patch(color='#FF9800', label='Noticeable (-0.05 <= ΔARI < 0)'),
    Patch(color='#4CAF50', label='Positive effect (more iterations)'),
]
ax.legend(handles=legend_elems, fontsize=9)

plt.tight_layout()
plt.show()

## 9. Quick preset selection

Three ready-made configurations on the speed-vs-quality tradeoff:

| Preset | n_iterations | n_bins | When |
|--------|-------------|--------|------|
| **fast** | 50 | 3 | EDA, first look |
| **balanced** | 200 | 3 | Production run |
| **thorough** | 500 | 4 | Final result |

In [ ]:
presets = {
    'fast':      dict(n_iterations=50,  n_bins=3),
    'balanced':  dict(n_iterations=200, n_bins=3),
    'thorough':  dict(n_iterations=500, n_bins=4),
}

preset_rows = []
for name, overrides in presets.items():
    params = {**BASE, **overrides}
    aris, times = [], []
    for s in range(5):
        clf = ForestClusterer(**params, clusterer=INNER(), random_state=s)
        t0 = time.perf_counter()
        lbl = clf.fit_predict(df0)
        times.append(time.perf_counter() - t0)
        aris.append(adjusted_rand_score(y0, lbl))
    preset_rows.append({
        'Preset': name,
        'n_iterations': overrides['n_iterations'],
        'n_bins': overrides['n_bins'],
        'ARI mean': round(np.mean(aris), 3),
        'ARI std':  round(np.std(aris),  3),
        'Time, s': round(np.mean(times), 2),
    })

df_pre = pd.DataFrame(preset_rows).set_index('Preset')
print(df_pre.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
colors_pre = ['#4CAF50', '#2196F3', '#9C27B0']
for i, (name, row) in enumerate(df_pre.iterrows()):
    ax.errorbar(row['Time, s'], row['ARI mean'], yerr=row['ARI std'],
                fmt='o', color=colors_pre[i], ms=16, lw=2, capsize=6,
                label=f'{name} (L={row.n_iterations}, K={row.n_bins})')
    ax.annotate(name, (row['Time, s'], row['ARI mean']),
                xytext=(8, -4), textcoords='offset points',
                fontsize=11, fontweight='bold', color=colors_pre[i])

ax.set_xlabel('fit_predict time, s', fontsize=11)
ax.set_ylabel('ARI (mean +/- std, 5 seeds)', fontsize=11)
ax.set_title(f'Three presets: speed vs quality tradeoff (n={N})',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## Final recommendations

| Parameter | Recommended | Main risk | Time impact |
|-----------|------------|-----------|-------------|
| `n_iterations` | **200** (start) → **500** (final) | Too low → low ARI and high std | Linear ↑ |
| `n_bins` | **3** | >5 with small n → empty cells, ARI ≈ 0 | Negligible |
| `n_features` | **`'sqrt'`** | M=1 → iterations carry little information | Negligible |
| `quantile_cuts` | **`True`** with outliers | `False` + outliers → broken bins, ΔARI −30%+ | Negligible |
| `corr_threshold` | **0.9** | <0.8 → ecological correlation | Negligible |
| `n_jobs` | **`-1`** | joblib overhead at n < 1000 | Linear ↓ per core |
| downstream | **KMeans(emb)** at n > 10K | AgglClust precomputed → O(n²) memory | O(n²) vs O(n) |

**Tuning order:**
1. `quantile_cuts=True` — always with unknown data (free of charge)
2. `n_iterations` — main lever: start with 50–100, final run 300–500
3. `corr_threshold=0.9` — if features may be correlated
4. `n_bins` and `n_features` — defaults (`3` and `'sqrt'`) rarely need changing